### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [3]:
account = Account.get("Ed")
account

Account(name='ed', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [4]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9861.724, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 46.092, "timestamp": "2026-02-11 10:31:51", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-11 10:31:51", 9873.724]], "total_portfolio_value": 9873.724, "total_profit_loss": -126.27599999999984}'

In [5]:
account.report()

'{"name": "ed", "balance": 9861.724, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 46.092, "timestamp": "2026-02-11 10:31:51", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-11 10:31:51", 9873.724], ["2026-02-11 10:31:58", 10053.724]], "total_portfolio_value": 10053.724, "total_profit_loss": 53.72400000000016}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 46.092,
  'timestamp': '2026-02-11 10:31:51',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [7]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [8]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='buy_shares', 

In [9]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [ ]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Ed, your current cash balance is $9,861.72. In your holdings, you have 3 shares of Amazon (AMZN). If you need any further assistance or want to make any transactions, please let me know!

[non-fatal] Tracing client error 400: {
  "error": {
    "message": "Invalid type for 'data[2].span_data.result': expected an array of strings, but got null instead.",
    "type": "invalid_request_error",
    "param": "data[2].span_data.result",
    "code": "invalid_type"
  }
}


### Now let's build our own MCP Client

In [11]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='buy_shares', ti

In [12]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Hi Ed, your current account balance is $9,861.72. Is there anything else you would like to know or do with your account?

In [13]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 9861.724, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 46.092, "timestamp": "2026-02-11 10:31:51", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-11 10:31:51", 9873.724], ["2026-02-11 10:31:58", 10053.724], ["2026-02-11 10:40:09", 10035.724]], "total_portfolio_value": 10035.724, "total_profit_loss": 35.72400000000016}


In [14]:
from accounts import Account
Account.get("ed").report()

'{"name": "ed", "balance": 9861.724, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 46.092, "timestamp": "2026-02-11 10:31:51", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-11 10:31:51", 9873.724], ["2026-02-11 10:31:58", 10053.724], ["2026-02-11 10:40:09", 10035.724], ["2026-02-11 10:40:12", 10122.724]], "total_portfolio_value": 10122.724, "total_profit_loss": 122.72400000000016}'

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>

In [19]:
# Set up params for both MCP servers
accounts_params = {"command": "uv", "args": ["run", "accounts_server.py"]}
date_params = {"command": "uv", "args": ["run", "date_server.py"]}

instructions = """You are a helpful assistant that can:
1. Manage accounts for clients (check balance, holdings, buy/sell shares)
2. Tell the current date

Always be helpful and use the appropriate tools to answer questions."""

model = "gpt-4o-mini"

# Use both MCP servers together
async with MCPServerStdio(params=accounts_params, client_session_timeout_seconds=30) as accounts_server:
    async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as date_server:
        agent = Agent(
            name="multi_tool_agent", 
            instructions=instructions,
            model=model,
            mcp_servers=[accounts_server, date_server]  # Both servers!
        )
        
        # Test with a request that uses both servers
        request = "What is today's date, and what is Ed's account balance?"
        
        with trace("multi_tool_agent"):
            result = await Runner.run(agent, request)
        display(Markdown(result.final_output))



Today's date is **February 11, 2026**, and Ed's account balance is **$9,861.72**.

In [21]:
# Interactive version with user input
# Run this cell, type your question, and press Enter

request = input("Enter your question: ")

if request.strip():
    async with MCPServerStdio(params=accounts_params, client_session_timeout_seconds=30) as accounts_server:
        async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as date_server:
            agent = Agent(
                name="interactive_agent", 
                instructions=instructions,
                model=model,
                mcp_servers=[accounts_server, date_server]
            )
            
            print(f"\nProcessing: {request}\n")
            with trace("interactive_agent"):
                result = await Runner.run(agent, request)
            display(Markdown(f"**Response:**\n\n{result.final_output}"))
else:
    print("No question entered.")


Processing: can you buy 10 shares of tesla because I think it looks cool



**Response:**

Before proceeding to buy 10 shares of Tesla, could you please provide the name of the account holder? Additionally, do you have a specific rationale that you would like me to include for the purchase?

In [22]:
# Continuous interactive loop - keep asking questions until you type 'exit'

print("Interactive Agent (Date + Accounts)")
print("Type 'exit' or 'quit' to stop\n")

while True:
    request = input(">> ")
    
    if request.lower().strip() in ('exit', 'quit', 'q'):
        print("Goodbye!")
        break
    
    if not request.strip():
        continue
    
    async with MCPServerStdio(params=accounts_params, client_session_timeout_seconds=30) as accounts_server:
        async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as date_server:
            agent = Agent(
                name="interactive_agent", 
                instructions=instructions,
                model=model,
                mcp_servers=[accounts_server, date_server]
            )
            
            with trace("interactive_agent"):
                result = await Runner.run(agent, request)
            display(Markdown(f"**Response:**\n\n{result.final_output}\n\n---"))

Interactive Agent (Date + Accounts)
Type 'exit' or 'quit' to stop



**Response:**

Ed currently has a balance of **$9,581.164**. He already holds **10 shares of Tesla (TSLA)**. Since he can't buy more shares at this time, is there anything else you would like to do regarding his investments?

---

**Response:**

Today's date is February 11, 2026.

---

Goodbye!
